# Preparing Geothermal Potentials for Kenya

This notebook downloads and prepares Geothermal potentials for Kenya for Pypsa-Earth model.

## Import packages

In [ ]:
import os
import requests
import py7zr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import country_converter as coco
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
import rasterio
import geopandas as gpd
from shapely.geometry import Polygon
from cartopy import crs as ccrs


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 70)

## User input

In [ ]:
# Geothermal potenital at the Great Rift Valley, Kenya
geothermal_potential = 10000 # MW Kenya is endowed with huge geothermal potential with estimates in excess of 10,000MW due to its location along the Great Rift Valley of Eastern Africa
                              # Source: doi: 10.1016/j.proeps.2013.03.220. page 2

# PyPSA-Earth bus region distribution of Kenya
state_gdf = gpd.read_file('/home/alex-charly/SSD/Kenya/pypsa-earth/resources/Kenya/bus_regions/regions_onshore_elec_s_2.geojson')
##regions_onshore = gpd.read_file('/home/alex-charly/SSD/Kenya/pypsa-earth/resources/Kenya/shapes/country_shapes.geojson')
# Alternative Clustering is on. Therefore, cluster 2 is not right

# Coordinates of the Great Rift Valley, Kenya
# Area extracted from doi: 10.1016/j.proeps.2013.03.220, figure 1 (Geothermal-Area-Great-Rift-Valley-Kenya) via https://automeris.io/wpd/.
coo_barrier = [(37.60000, -2.40452)]
coo_namarunu = [(37.43000, -2.09974)]
coo_emuruangogolak = [(37.36667, -1.55671)]
coo_silali = [(37.26333, -1.23351)]
coo_baka = [(37.22333, -1.01474)]
coo_korosi = [(37.15333, -0.84022)]
coo_lakebaringo = [(37.09667, -0.64360)]
coo_arus = [(37.10000, -0.20934)]
coo_menengai = [(37.10000, 0.12399)]
coo_eburru = [(37.23000, 0.60816)]
coo_olkaria = [(37.26333, 0.85860)]
coo_longonot = [(37.42000, 0.89575)]
coo_suswa = [(37.35667, 1.14491)]
coo_lakemagadi = [(37.22000, 1.84925)]

coordinates = [
    coo_barrier[0],
    coo_namarunu[0],
    coo_emuruangogolak[0],
    coo_silali[0],
    coo_baka[0],
    coo_korosi[0],
    coo_lakebaringo[0],
    coo_arus[0],
    coo_menengai[0],
    coo_eburru[0],
    coo_olkaria[0],
    coo_longonot[0],
    coo_suswa[0],
    coo_lakemagadi[0]
]

# Set distance to make a polygon based on the given coordinate
buffer_distance = 0.01

## Preparation

In [ ]:
# Function to create a polygon with buffer
def create_polygon_with_buffer(coord, buffer_distance):
    return Polygon([
        (coord[0] - buffer_distance, coord[1] - buffer_distance),  # Bottom-left
        (coord[0] + buffer_distance, coord[1] - buffer_distance),  # Bottom-right
        (coord[0] + buffer_distance, coord[1] + buffer_distance),  # Top-right
        (coord[0] - buffer_distance, coord[1] + buffer_distance),  # Top-left
        (coord[0] - buffer_distance, coord[1] - buffer_distance)   # Closing the polygon
    ])

# Create polygons for all geothermal coordinates
polygons = [create_polygon_with_buffer(coord, buffer_distance) for coord in coordinates]

# Check if each polygon (geothermal area) is within any polygon in state_gdf (pypsa-earth region)
state_gdf['inside_polygon'] = state_gdf.geometry.apply(lambda x: any(poly.within(x) for poly in polygons))

In [ ]:
# Plot the geothermal coordinates in relation to pypsa-earth region
polygons_gdf = gpd.GeoDataFrame({'geometry': polygons}, crs=state_gdf.crs)
fig, ax = plt.subplots()
state_gdf.boundary.plot(ax=ax, color='black', linewidth=1)
polygons_gdf.boundary.plot(ax=ax, color='red', linewidth=1)
#regions_onshore.boundary.plot(ax=ax, color="black", linewidth=1)
plt.show()

In [ ]:
# Plot geothermal coordinates in relation to pypsa-earth region
gdf = state_gdf.copy()
# Create a new column 'color' based on the condition you specified
gdf['color'] = 'blue'
gdf.loc[gdf['inside_polygon'] == True, 'color'] = 'red'

# Plot the GeoDataFrame with colors
fig, ax = plt.subplots(figsize=(10, 10))

# Plot states with colors based on the 'color' column
gdf.plot(ax=ax, column='color', legend=True, cmap='coolwarm')
#regions_onshore.boundary.plot(ax=ax, color='black', linewidth=1)

# Set the title
plt.title('Great Rift Valley - Geothermal Area in Kenya')

# Set the legend labels using the 'name' attribute
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, title='Legend')

# Show the plot
plt.show()

In [ ]:
# Create a new column based on shares
true_count = state_gdf['inside_polygon'].sum() 

geothermal_exist_31_1_AC = 384 

state_gdf['potential [MWel]'] = state_gdf['inside_polygon'].apply(lambda x: (geothermal_potential - geothermal_exist_31_1_AC) / true_count if x else 0)
state_gdf.loc[state_gdf.name == "KE.31_1_AC", "potential [MWel]"] = geothermal_exist_31_1_AC 
state_gdf.to_csv('geothermal_potential.csv')

In [ ]:
# Plot the GeoDataFrame with colors
fig, ax = plt.subplots(figsize=(10, 10))

# Plot states with colors based on the 'color' column
state_gdf.plot(ax=ax, column='potential [MWel]', legend=True, cmap='coolwarm')

# Set the title
plt.title('Great Rift Valley - Geothermal Area in Kenya')

# Set the legend labels using the 'name' attribute
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, title='Legend')

# Show the plot
plt.show()